# 07 · Files & Python: HDB Prices and Your PDFs 🏠📄🐍

Now the bot works with **files**:

| What | How |
|---|---|
| `hdb_resale_prices.csv` — every HDB resale flat sold in the last 12 months (from data.gov.sg) | the **python tool** (`code_interpreter`) writes and runs Python on it |
| `my_flat_notes.pdf` — a couple's flat-hunting notes | a **file input** the AI reads |
| anything that changes, like news | **web search** |

Built-in tools like these are **run by OpenAI for you** — no Python function needed.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. A quick look at the data (pandas)

In [ ]:
import pandas as pd

hdb = pd.read_csv("hdb_resale_prices.csv")
print(len(hdb), "flats sold")
hdb.head()

That's a lot of rows to paste into a chat! Instead, we **upload** the file to OpenAI and let the AI analyse it with Python.

## 2. Upload the CSV

In [ ]:
hdb_file = client.files.create(file=open("hdb_resale_prices.csv", "rb"), purpose="user_data")

print(hdb_file.id)

## 3. The python tool (code interpreter) 🐍
We give the python tool a **container** (a little computer at OpenAI) with our file inside.

In [ ]:
python_tool = {"type": "code_interpreter", "container": {"type": "auto", "file_ids": [hdb_file.id]}}

response = client.responses.create(
    model=MODEL,
    instructions=bot + " Use the python tool to analyse hdb_resale_prices.csv.",
    tools=[python_tool],
    input="What's the median 4-room resale price in Tampines, Bedok and Pasir Ris?",
)

print(response.output_text)

In [ ]:
# peek at the Python code the AI wrote
for item in response.output:
    if item.type == "code_interpreter_call":
        print(item.code)

> 💰 Each python tool session costs a few cents, and it stays open for 20 minutes after its last use.

## 4. Send a PDF 📄
Open `my_flat_notes.pdf` in the file list to see what's inside. Then upload it and send it as an `input_file`.

In [ ]:
notes_file = client.files.create(file=open("my_flat_notes.pdf", "rb"), purpose="user_data")

response = client.responses.create(
    model=MODEL,
    instructions=bot,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Summarise my flat-hunting notes in 3 lines."},
            {"type": "input_file", "file_id": notes_file.id},
        ],
    }],
)

print(response.output_text)

## 5. Put them together 🤝
The PDF says **what we want**, the CSV says **what flats really cost**.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=bot + " Use the python tool to analyse hdb_resale_prices.csv.",
    tools=[python_tool],
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Based on my notes, which of our towns fit our budget? How many flats sold that match everything we want?"},
            {"type": "input_file", "file_id": notes_file.id},
        ],
    }],
)

print(response.output_text)

## 6. Web search 🔍
For anything that changes — news, policies, opening hours.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=bot,
    tools=[{"type": "web_search"}],
    input="Any recent news about HDB resale prices?",
)

print(response.output_text)

## ✏️ Exercises
1. Ask the python tool: *"Which town has the cheapest 5-room flats?"* and *"How did prices change month by month?"*
2. Change the budget in your question to $550,000 — what changes?
3. 🌶️ Challenge: download another dataset from **data.gov.sg** (e.g. COE prices), upload it, and ask about it.

In [ ]:
# your code here